<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/BERTCustomModelPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install transformers

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset

In [7]:
autotoken = AutoTokenizer.from_pretrained('bert-base-uncased')
autotoken.vocab_size

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522

In [8]:
raw_datasets = load_dataset('imdb')

train_Dataset = raw_datasets['train']
val_Dataset = raw_datasets['test']

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [14]:
embed_dim=256
vocab_size=autotoken.vocab_size
max_len=200
num_heads=8
ff_dim= 4 * embed_dim
num_layers=4
num_classes=2

In [9]:
autotoken = AutoTokenizer.from_pretrained('bert-base-uncased')

In [15]:
class IMDBDataset(Dataset):
  def __init__(self,text,labels,max_len):
    self.text = text
    self.tokenizer = autotoken
    self.lables = labels
    self.max_len = max_len

  def __len__(self):
    return len(self.text)

  def __getitem__(self,idx):
    # clean the text, real world has messy or nan
    item = str(self.text[idx])

    encoding = self.tokenizer(text=item,padding='max_length',max_length=self.max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

    return {
        'input_ids':encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(self.lables[idx],dtype=torch.long)
    }


In [16]:
train_data = IMDBDataset(text=train_Dataset['text'],labels=train_Dataset['label'],max_len=max_len)
val_data = IMDBDataset(text=val_Dataset['text'],labels=val_Dataset['label'],max_len=max_len)

In [17]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,pin_memory=True)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=False,pin_memory=True)

In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,ff_dim) -> None:
    super().__init__()
    self.attention = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads,dropout=0.2)

    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim,out_features=ff_dim),
        nn.GELU(),
        nn.Linear(in_features=ff_dim,out_features=embed_dim),
        nn.Dropout(0.3)
    )

    self.layernorm1 = nn.LayerNorm(normalized_shape=embed_dim)
    self.layernorm2 = nn.LayerNorm(normalized_shape=embed_dim)

  def forward(self,inputtext):
    att,_ = self.attention(inputtext,inputtext,inputtext)
    merged_att = self.layernorm1(inputtext + att)
    mlp_output = self.mlp(merged_att)
    return self.layernorm2(merged_att + mlp_output)


In [ ]:
class WordEmbedding(nn.Module):
  def __init__(self,embed_dim,vocab_size) -> None:
    super().__init__()

    self.wordEmbed = nn.Embedding(num_embeddings=vocab_size,embedding_dim=embed_dim)
  def forward(self,x):
    return self.wordEmbed(x)

In [ ]:
class NLPTransformer(nn.Module):
  def __init__(self,embed_dim,vocab_size,max_len,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.wordEmbed = WordEmbedding(embed_dim=embed_dim,vocab_size=vocab_size)
    self.positionEmbed = nn.Parameter(torch.zeros(1,max_len,embed_dim))

    self.transform_layers = nn.ModuleList([
        TransformerBlock(embed_dim=embed_dim,num_heads=num_heads,ff_dim=ff_dim)
        for _ in range(num_layers)
    ])
    self.dropout = nn.Dropout(0.3)
    self.output_layer = nn.Linear(in_features=embed_dim,out_features=num_classes)

  def forward(self,x,mask):
    x = self.wordEmbed(x) + self.positionEmbed
    x = self.dropout(x)

    for transformer_layer in self.transform_layers:
      x = transformer_layer(x)

    # 3. THE SMART MEAN (Replacing x = x.mean(dim=1))
    # We need to make the mask (Batch, 512) match x (Batch, 512, 128)
    mask = mask.unsqueeze(-1) # shape becomes [Batch, 512, 1]
    x = x * mask
    # Sum only the real words and divide by the count of real words
    x = x.sum(dim=1) / mask.sum(dim=1)
    return self.output_layer(x)

In [19]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model = NLPTransformer(embed_dim=embed_dim,
                       vocab_size=vocab_size,
                       max_len=max_len,
                       num_heads=num_heads,
                       ff_dim=ff_dim,
                       num_layers=num_layers,
                       num_classes=num_classes).to(device)

In [ ]:
optimizer = optim.Adam(params=model.parameters(),lr=5e-5,weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
epochs = 12

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for batch in train_loader:
     # 1. Unpack all three items from your Dictionary
    input_ids = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()

    output = model(input_ids,mask)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = torch.argmax(input=output,dim=1)
    train_correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total
  train_losses = train_loss / len(train_loader)

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for batch in val_loader:
     # 1. Unpack all three items from your Dictionary
     input_ids = batch['input_ids'].to(device)
     mask = batch['attention_mask'].to(device)
     labels = batch['labels'].to(device)

     output = model(input_ids,mask)
     loss = loss_fn(output,labels)

     val_loss += loss.item()
     pred = torch.argmax(input=output,dim=1)
     val_correct += (pred == labels).sum().item()
     val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_losses = val_loss / len(val_loader)

    print(f'\nEpochs: {epoch+1}/{epochs}...')
    print(f'Train_acc: {train_accuracy} | Train_loss: {train_losses}')
    print(f'Val_acc: {val_accuracy} | Val_loss: {val_losses}')
    print('==============================================================')


Epochs: 1/12...
Train_acc: 0.62888 | Train_loss: 0.636312956052363
Val_acc: 0.57908 | Val_loss: 0.7394038527594198

Epochs: 2/12...
Train_acc: 0.70832 | Train_loss: 0.5673861476161596
Val_acc: 0.615 | Val_loss: 0.6927182602760432

Epochs: 3/12...
Train_acc: 0.73556 | Train_loss: 0.5328892608890143
Val_acc: 0.60564 | Val_loss: 0.715259957839461

Epochs: 4/12...
Train_acc: 0.759 | Train_loss: 0.4995370870432281
Val_acc: 0.62288 | Val_loss: 0.7111828543264848

Epochs: 5/12...
Train_acc: 0.77696 | Train_loss: 0.4709920719685152
Val_acc: 0.62428 | Val_loss: 0.7337701835900622

Epochs: 6/12...
Train_acc: 0.79748 | Train_loss: 0.4410296150333131
Val_acc: 0.62832 | Val_loss: 0.7554488944656709

Epochs: 7/12...
Train_acc: 0.81044 | Train_loss: 0.41886665592032013
Val_acc: 0.62072 | Val_loss: 0.7920810418284457

Epochs: 8/12...
Train_acc: 0.81876 | Train_loss: 0.40289490666154704
Val_acc: 0.63248 | Val_loss: 0.7640015050730742

Epochs: 9/12...
Train_acc: 0.82528 | Train_loss: 0.3864405976079614

**PreTrained Model**

In [4]:
class BertPretrained(nn.Module):
  def __init__(self,num_classes) -> None:
    super().__init__()

    self.bert = AutoModel.from_pretrained('bert-base-uncased')
    self.dropout = nn.Dropout(0.2)
    self.classifier = nn.LazyLinear(num_classes)

  def forward(self,input_ids,attentionmask):
    output = self.bert(input_ids=input_ids,attention_mask=attentionmask)

    # Grab the 'meaning' of every word
    x = output.last_hidden_state # [Batch, 200, 768]


    # USE YOUR SMART MEAN (The one we built together!)
    mask = attentionmask.unsqueeze(-1)
    x = (x * mask).sum(dim=1) / mask.sum(dim=1)

    # Final Guess
    return self.classifier(self.dropout(x))

In [ ]:
# 1. Initialize Model
bert_model = BertPretrained(num_classes=2).to(device)

# 2. Optimizer (The "Gentle" Settings)
optimizer = torch.optim.AdamW(
    bert_model.parameters(),
    lr=2e-5,          # <--- Critical: 0.00002
    weight_decay=0.01 # <--- Standard for Pre-trained models
)
loss_fn = nn.CrossEntropyLoss()

In [23]:
epochs = 5

for epoch in range(epochs):
  bert_model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for batch in train_loader:
     # 1. Unpack all three items from your Dictionary
    input_ids = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()

    output = bert_model(input_ids,mask)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = torch.argmax(input=output,dim=1)
    train_correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total
  train_losses = train_loss / len(train_loader)

  bert_model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for batch in val_loader:
     # 1. Unpack all three items from your Dictionary
     input_ids = batch['input_ids'].to(device)
     mask = batch['attention_mask'].to(device)
     labels = batch['labels'].to(device)

     output = bert_model(input_ids,mask)
     loss = loss_fn(output,labels)

     val_loss += loss.item()
     pred = torch.argmax(input=output,dim=1)
     val_correct += (pred == labels).sum().item()
     val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_losses = val_loss / len(val_loader)

    print(f'\nEpochs: {epoch+1}/{epochs}...')
    print(f'Train_acc: {train_accuracy} | Train_loss: {train_losses}')
    print(f'Val_acc: {val_accuracy} | Val_loss: {val_losses}')
    print('==============================================================')


Epochs: 1/5...
Train_acc: 0.87496 | Train_loss: 0.28763754497213134
Val_acc: 0.91244 | Val_loss: 0.2207734445949583


KeyboardInterrupt: 